A make_features.py that reads raw CSVs and emits a clean, engineered, documented
training table — the 'Feature Store' node in the data-flow diagram. Every later week imports it.

In [6]:
#import libraries
import pandas as pd

In [7]:
#read datset
df = pd.read_csv("orders.csv")
print(df.head())

   order_id  user_id  weight_kg  distance_km  express_flag  quantity    price  \
0         0     4647       1.23        303.0         False         3   3823.0   
1         1     3774       1.54         68.0         False         1   3539.0   
2         2     2876       2.27        216.0         False         3  18507.0   
3         3     3155       5.26        354.0         False         5   5797.0   
4         4     2066       3.57         92.0          True         2   1004.0   

   shipping_cost  
0          222.0  
1          119.0  
2          171.0  
3          218.0  
4          203.0  


In [8]:
#inspect dataset
print(df.info())
print(df.describe())
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       40000 non-null  int64  
 1   user_id        40000 non-null  int64  
 2   weight_kg      40000 non-null  float64
 3   distance_km    40000 non-null  float64
 4   express_flag   40000 non-null  bool   
 5   quantity       40000 non-null  int64  
 6   price          40000 non-null  float64
 7   shipping_cost  40000 non-null  float64
dtypes: bool(1), float64(4), int64(3)
memory usage: 2.2 MB
None
          order_id       user_id     weight_kg   distance_km      quantity  \
count  40000.00000  40000.000000  40000.000000  40000.000000  40000.000000   
mean   19999.50000   2503.548975      3.987978    180.376025      2.992400   
std    11547.14972   1444.353971      2.844421    104.676651      1.419029   
min        0.00000      0.000000      0.010000      2.000000      1.000000  

In [9]:
#handle missing values
num_cols = df.select_dtypes(include=["number"]).columns
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

In [10]:
#categorical columns
cat_cols = df.select_dtypes(include=["object"]).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [11]:
#remove duplicates
df = df.drop_duplicates()

In [12]:
#feature engineering

# 1. Average price per item
df["price_per_item"] = df["price"] / df["quantity"]

# 2. Weight per item
df["weight_per_item"] = df["weight_kg"] / df["quantity"]

# 3. Shipping cost per kilometer
df["shipping_per_km"] = df["shipping_cost"] / df["distance_km"]

# 4. Shipping cost as percentage of price
df["shipping_price_ratio"] = (
    df["shipping_cost"] / df["price"]
) * 100

# 5. Heavy package indicator
df["heavy_package"] = (df["weight_kg"] > 5).astype(int)

# 6. Long-distance delivery
df["long_distance"] = (df["distance_km"] > 200).astype(int)

# 7. Bulk order
df["bulk_order"] = (df["quantity"] >= 5).astype(int)

In [13]:
#new feature
print(df.head())

   order_id  user_id  weight_kg  distance_km  express_flag  quantity    price  \
0         0     4647       1.23        303.0         False         3   3823.0   
1         1     3774       1.54         68.0         False         1   3539.0   
2         2     2876       2.27        216.0         False         3  18507.0   
3         3     3155       5.26        354.0         False         5   5797.0   
4         4     2066       3.57         92.0          True         2   1004.0   

   shipping_cost  price_per_item  weight_per_item  shipping_per_km  \
0          222.0     1274.333333         0.410000         0.732673   
1          119.0     3539.000000         1.540000         1.750000   
2          171.0     6169.000000         0.756667         0.791667   
3          218.0     1159.400000         1.052000         0.615819   
4          203.0      502.000000         1.785000         2.206522   

   shipping_price_ratio  heavy_package  long_distance  bulk_order  
0              5.806958 

In [16]:
#saving new features
df.to_csv("orders_features.csv", index=False)
print("feature store created")

feature store created


In [18]:
#details about orders_features
df.to_csv("orders_features.csv", index=False)
print(df.shape)
print(df.columns.tolist())
print(df.head())

(40000, 15)
['order_id', 'user_id', 'weight_kg', 'distance_km', 'express_flag', 'quantity', 'price', 'shipping_cost', 'price_per_item', 'weight_per_item', 'shipping_per_km', 'shipping_price_ratio', 'heavy_package', 'long_distance', 'bulk_order']
   order_id  user_id  weight_kg  distance_km  express_flag  quantity    price  \
0         0     4647       1.23        303.0         False         3   3823.0   
1         1     3774       1.54         68.0         False         1   3539.0   
2         2     2876       2.27        216.0         False         3  18507.0   
3         3     3155       5.26        354.0         False         5   5797.0   
4         4     2066       3.57         92.0          True         2   1004.0   

   shipping_cost  price_per_item  weight_per_item  shipping_per_km  \
0          222.0     1274.333333         0.410000         0.732673   
1          119.0     3539.000000         1.540000         1.750000   
2          171.0     6169.000000         0.756667        